# Notebook 1: MLP for Regression from Scratch (NumPy)

In this notebook, we will build a Multi-Layer Perceptron (MLP) from scratch using only `numpy`.
Our goal is to understand the internal mechanics of a neural network: forward pass, backpropagation, activation functions, loss calculation, and optimization algorithms.

**What you will learn**
- how the forward pass and backpropagation work, layer by layer
- how to check that your gradients are correct
- how SGD, momentum and Adam differ
- how to detect overfitting with a validation set, and what L1 and L2 regularization do about it

**Runtime:** about 10 seconds on a laptop.

## Task: Predicting Distance from Redshift

We will use a toy dataset inspired by astrophysics: predicting the distance ($D$) of a galaxy based on its redshift ($z$).
The underlying relationship we simulate is roughly quadratic, mimicking a simplified cosmological expansion model:

$$ D \approx \frac{c}{H_0} z + \text{higher order terms} $$

In the code below we use $c/H_0 = 3000\,\mathrm{Mpc}$ (i.e. $H_0 = 100\,\mathrm{km\,s^{-1}\,Mpc^{-1}}$) and a quadratic term $500\,\mathrm{Mpc} \cdot z^2$.
We will add some Gaussian noise to simulate measurement errors.

We generate two datasets: a **training set** which is used to fit the network, and an independent **validation set** which is only used to check how well the network performs on data it has not seen. With real data you would split your dataset into two parts instead. Since we simulate the data ourselves, we also know the true relation and can compare our network to it at the end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

noise_std = 500   # Mpc

def true_relation(z):
    # D = 3000*z + 500*z^2 (in Mpc)
    return 3000 * z + 500 * z**2

# 1. Generate Data
def generate_data(n_samples=200):
    z = np.random.uniform(0, 3, n_samples)
    noise = np.random.normal(0, noise_std, n_samples)
    D = true_relation(z) + noise

    return z.reshape(-1, 1), D.reshape(-1, 1)

# Training data: used to fit the weights
X_train, y_train = generate_data(200)
# Validation data: never used for fitting, only to check how well the model generalizes
X_val, y_val = generate_data(200)

# Normalize data for easier training (Standard Scaling)
# The mean and standard deviation are computed from the training data only
X_mean, X_std = X_train.mean(), X_train.std()
y_mean, y_std = y_train.mean(), y_train.std()

X_train_norm = (X_train - X_mean) / X_std
y_train_norm = (y_train - y_mean) / y_std
X_val_norm = (X_val - X_mean) / X_std
y_val_norm = (y_val - y_mean) / y_std

z_plot = np.linspace(0, 3, 100)
plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, s=10, label='Training data')
plt.scatter(X_val, y_val, s=10, label='Validation data', alpha=0.5)
plt.plot(z_plot, true_relation(z_plot), color='black', ls='--', label='True relation')
plt.xlabel('Redshift (z)')
plt.ylabel('Distance (Mpc)')
plt.title('Toy Dataset: Redshift vs Distance')
plt.legend()
plt.show()

## 1. Building Blocks of a Neural Network

A neural network consists of layers. Each layer performs a linear transformation followed by a non-linear activation.

### Linear Layer

The linear layer computes:
$$ Z = X W + b $$

Where:
- $X$ is the input matrix of shape $(N, D_{in})$.
- $W$ is the weight matrix of shape $(D_{in}, D_{out})$.
- $b$ is the bias vector of shape $(1, D_{out})$.
- $Z$ is the output of shape $(N, D_{out})$.

During backpropagation, we need to compute gradients with respect to $X$, $W$, and $b$ given the gradient of the loss with respect to $Z$ ($dZ$).

$$ dX = dZ \cdot W^T $$
$$ dW = X^T \cdot dZ $$
$$ db = \sum dZ $$

In [ ]:
class Layer:
    def forward(self, input):
        pass
    def backward(self, output_gradient):
        pass

class Linear(Layer):
    def __init__(self, input_dim, output_dim):
        # He initialization (suited for ReLU-like activations)
        self.weights = np.random.randn(input_dim, output_dim) * np.sqrt(2. / input_dim)
        self.bias = np.zeros((1, output_dim))
        
        # Storage for gradients (needed for optimizers)
        self.grad_weights = None
        self.grad_bias = None

    def forward(self, input):
        self.input = input
        return np.dot(self.input, self.weights) + self.bias

    def backward(self, output_gradient):
        # Compute gradients
        self.grad_weights = np.dot(self.input.T, output_gradient)
        self.grad_bias = np.sum(output_gradient, axis=0, keepdims=True)
        
        # Return gradient w.r.t input to pass to previous layer
        return np.dot(output_gradient, self.weights.T)

### Activation Functions

We will implement **ReLU**, **LeakyReLU**, and **Sigmoid**.

**ReLU (Rectified Linear Unit):**
$$ f(x) = \max(0, x) $$
$$ f'(x) = 1 \text{ if } x > 0, \text{ else } 0 $$

**LeakyReLU:**
$$ f(x) = \max(\alpha x, x) \quad (\text{usually } \alpha=0.01) $$

**Sigmoid:**
$$ f(x) = \frac{1}{1 + e^{-x}} $$
$$ f'(x) = f(x) (1 - f(x)) $$

In [ ]:
class Activation(Layer):
    def __init__(self, activation, activation_prime):
        self.activation = activation
        self.activation_prime = activation_prime

    def forward(self, input):
        self.input = input
        return self.activation(self.input)

    def backward(self, output_gradient):
        return output_gradient * self.activation_prime(self.input)

class ReLU(Activation):
    def __init__(self):
        relu = lambda x: np.maximum(0, x)
        relu_prime = lambda x: (x > 0).astype(float)
        super().__init__(relu, relu_prime)

class LeakyReLU(Activation):
    def __init__(self, alpha=0.01):
        self.alpha = alpha
        leaky_relu = lambda x: np.where(x > 0, x, x * alpha)
        leaky_relu_prime = lambda x: np.where(x > 0, 1, alpha)
        super().__init__(leaky_relu, leaky_relu_prime)

class Sigmoid(Activation):
    def __init__(self):
        def sigmoid(x):
            return 1 / (1 + np.exp(-x))
        def sigmoid_prime(x):
            s = sigmoid(x)
            return s * (1 - s)
        super().__init__(sigmoid, sigmoid_prime)

### Loss Function with Regularization

We use Mean Squared Error (MSE) for regression:
$$ L = \frac{1}{N} \sum (y_{pred} - y_{true})^2 $$

**Regularization (L1/L2):**
We can add a penalty to the loss to prevent overfitting.

- **L2 (Ridge):** $\lambda_2 \sum w^2$, with gradient $2 \lambda_2 w$
- **L1 (Lasso):** $\lambda_1 \sum |w|$, with gradient $\lambda_1 \,\mathrm{sign}(w)$

The penalty only depends on the weights and not on the data, so we do not need to backpropagate it through the network. Instead, we simply add its gradient to the weight gradients in the optimization step (see the optimizers below). As is common, we only regularize the weights and not the biases.

In [ ]:
class MSE:
    def forward(self, pred, true):
        return np.mean((pred - true) ** 2)

    def backward(self, pred, true):
        return 2 * (pred - true) / true.size

### Checking Backpropagation: the Gradient Check

Every `backward` method above applies the chain rule to one layer: it receives $\partial L / \partial(\text{output})$ and returns $\partial L / \partial(\text{input})$. Calling the `backward` methods in reverse order therefore propagates the gradient of the loss from the output back to every weight.

It is easy to make a mistake in such formulas (a missing transpose, a wrong sign), and a network with slightly wrong gradients often still trains, just worse. The standard test is to compare with a numerical derivative. For a single parameter $w$:

$$ \frac{\partial L}{\partial w} \approx \frac{L(w + \epsilon) - L(w - \epsilon)}{2 \epsilon} $$

This needs two forward passes *per parameter*, which is far too slow for training, but perfect for testing. If both agree to many digits, our backpropagation is correct.

In [ ]:
def predict(network, X):
    """Forward pass: a network is just a list of layers which are applied one after the other."""
    output = X
    for layer in network:
        output = layer.forward(output)
    return output

def numerical_gradient(network, loss_fn, X, y, param, eps=1e-6):
    """Central finite differences: change one entry of `param` at a time and see how the loss reacts."""
    grad = np.zeros_like(param)
    for index in np.ndindex(*param.shape):
        original_value = param[index]
        param[index] = original_value + eps
        loss_plus = loss_fn.forward(predict(network, X), y)
        param[index] = original_value - eps
        loss_minus = loss_fn.forward(predict(network, X), y)
        param[index] = original_value
        grad[index] = (loss_plus - loss_minus) / (2 * eps)
    return grad

np.random.seed(0)
X_check, y_check = np.random.randn(5, 1), np.random.randn(5, 1)
loss_fn = MSE()

for activation in (ReLU, LeakyReLU, Sigmoid):
    test_network = [Linear(1, 4), activation(), Linear(4, 4), activation(), Linear(4, 1)]

    # Gradients from backpropagation: forward pass, then backward pass in reverse order
    output = predict(test_network, X_check)
    grad = loss_fn.backward(output, y_check)
    for layer in reversed(test_network):
        grad = layer.backward(grad)

    # Compare with the numerical gradients for all weights and biases
    max_difference = 0.
    for layer in test_network:
        if isinstance(layer, Linear):
            for param, grad_backprop in ((layer.weights, layer.grad_weights), (layer.bias, layer.grad_bias)):
                grad_numerical = numerical_gradient(test_network, loss_fn, X_check, y_check, param)
                max_difference = max(max_difference, np.abs(grad_numerical - grad_backprop).max())
    print(f'{activation.__name__:10s} largest difference between backpropagation and numerical gradient: {max_difference:.1e}')

## 2. Optimizers

Gradient descent algorithms update the weights to minimize the loss.

### SGD (Stochastic Gradient Descent)
$$ W = W - \eta \cdot \nabla W $$

"Stochastic" refers to estimating the gradient from a random mini-batch of the data. Our dataset is tiny, so below we simply use the full dataset in every step (mini-batches follow in Notebook 2).

### Momentum
Accumulates a velocity vector:
$$ v_t = \gamma v_{t-1} + \eta \nabla W $$
$$ W = W - v_t $$

### RMSProp
Scales the learning rate by a running average of the squared magnitudes of the gradients.

### Adam (Adaptive Moment Estimation)
Combines Momentum and RMSProp. It computes adaptive learning rates for each parameter.
$$ m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t $$
$$ v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2 $$
$$ \hat{m}_t = m_t / (1 - \beta_1^t) $$
$$ \hat{v}_t = v_t / (1 - \beta_2^t) $$
$$ W = W - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} $$

Below we implement SGD (with optional momentum) and Adam. Both accept the regularization strengths `l1_lambda` and `l2_lambda`.

In [ ]:
class Optimizer:
    def __init__(self, l1_lambda=0.0, l2_lambda=0.0):
        self.l1_lambda = l1_lambda
        self.l2_lambda = l2_lambda

    def weight_gradient(self, layer):
        # Gradient of the data loss plus the gradient of the L1/L2 penalty
        return (layer.grad_weights
                + 2 * self.l2_lambda * layer.weights
                + self.l1_lambda * np.sign(layer.weights))

    def step(self, layers):
        pass

class SGD(Optimizer):
    def __init__(self, learning_rate=0.01, momentum=0.0, l1_lambda=0.0, l2_lambda=0.0):
        super().__init__(l1_lambda, l2_lambda)
        self.lr = learning_rate
        self.momentum = momentum
        self.velocity = {}

    def step(self, layers):
        for layer in layers:
            if not isinstance(layer, Linear): continue

            if layer not in self.velocity:
                self.velocity[layer] = {'w': np.zeros_like(layer.weights), 'b': np.zeros_like(layer.bias)}
            v = self.velocity[layer]

            # Update velocity (for momentum=0 this is plain gradient descent)
            v['w'] = self.momentum * v['w'] + self.lr * self.weight_gradient(layer)
            v['b'] = self.momentum * v['b'] + self.lr * layer.grad_bias

            # Update parameters
            layer.weights -= v['w']
            layer.bias -= v['b']

class Adam(Optimizer):
    def __init__(self, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8, l1_lambda=0.0, l2_lambda=0.0):
        super().__init__(l1_lambda, l2_lambda)
        self.lr = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = {}
        self.v = {}
        self.t = 0

    def step(self, layers):
        self.t += 1
        for layer in layers:
            if not isinstance(layer, Linear): continue
            
            if layer not in self.m:
                self.m[layer] = {'w': np.zeros_like(layer.weights), 'b': np.zeros_like(layer.bias)}
                self.v[layer] = {'w': np.zeros_like(layer.weights), 'b': np.zeros_like(layer.bias)}
            
            grad_w = self.weight_gradient(layer)
            grad_b = layer.grad_bias

            # Update first moment (m)
            self.m[layer]['w'] = self.beta1 * self.m[layer]['w'] + (1 - self.beta1) * grad_w
            self.m[layer]['b'] = self.beta1 * self.m[layer]['b'] + (1 - self.beta1) * grad_b

            # Update second moment (v)
            self.v[layer]['w'] = self.beta2 * self.v[layer]['w'] + (1 - self.beta2) * grad_w**2
            self.v[layer]['b'] = self.beta2 * self.v[layer]['b'] + (1 - self.beta2) * grad_b**2

            # Bias correction
            m_hat_w = self.m[layer]['w'] / (1 - self.beta1**self.t)
            v_hat_w = self.v[layer]['w'] / (1 - self.beta2**self.t)
            m_hat_b = self.m[layer]['b'] / (1 - self.beta1**self.t)
            v_hat_b = self.v[layer]['b'] / (1 - self.beta2**self.t)

            # Update parameters
            layer.weights -= self.lr * m_hat_w / (np.sqrt(v_hat_w) + self.epsilon)
            layer.bias -= self.lr * m_hat_b / (np.sqrt(v_hat_b) + self.epsilon)

## 3. Training the Model

We will build a simple network with two hidden layers: `Input -> Linear -> LeakyReLU -> Linear -> LeakyReLU -> Linear -> Output`.

One training step (here: one *epoch*, since we always use the full training set) consists of a forward pass, the loss, a backward pass and an optimizer step. We put the loop into a function since we will train several networks below. In every epoch we also evaluate the loss on the validation data.

The loss printed during training is the plain MSE (without the regularization penalty). Since the noise in the data cannot be predicted, the MSE of a good model should end up close to the variance of the noise, and not at zero.

In [ ]:
def make_network(hidden_dim=16, seed=0):
    """Input -> Linear -> LeakyReLU -> Linear -> LeakyReLU -> Linear -> Output"""
    np.random.seed(seed)    # same initial weights every time (useful for comparisons later)
    return [
        Linear(1, hidden_dim),
        LeakyReLU(),
        Linear(hidden_dim, hidden_dim),
        LeakyReLU(),
        Linear(hidden_dim, 1)
    ]

def train(network, optimizer, X_train, y_train, X_val, y_val, epochs=2000, print_every=None):
    loss_fn = MSE()
    train_loss_history, val_loss_history = [], []

    for epoch in range(epochs):
        # Validation loss: forward pass only, the validation data is never used to update the weights
        val_loss = loss_fn.forward(predict(network, X_val), y_val)
        val_loss_history.append(val_loss)

        # Forward Pass
        output = predict(network, X_train)

        # Loss
        loss = loss_fn.forward(output, y_train)
        train_loss_history.append(loss)

        # Backward Pass
        grad = loss_fn.backward(output, y_train)
        for layer in reversed(network):
            grad = layer.backward(grad)

        # Optimization Step
        optimizer.step(network)

        if print_every is not None and epoch % print_every == 0:
            print(f'Epoch {epoch}, Training Loss: {loss:.5f}, Validation Loss: {val_loss:.5f}')

    return np.array(train_loss_history), np.array(val_loss_history)

network = make_network()
# Try switching between SGD and Adam
# optimizer = SGD(learning_rate=0.01, momentum=0.9, l2_lambda=0.001)
optimizer = Adam(learning_rate=0.01)

train_loss, val_loss = train(network, optimizer, X_train_norm, y_train_norm, X_val_norm, y_val_norm,
                             epochs=2000, print_every=200)

# The noise sets a lower limit for the MSE: in normalized units it is (500 Mpc / y_std)^2
noise_level = (noise_std / y_std) ** 2

# Visualize Loss
plt.plot(train_loss, label='Training')
plt.plot(val_loss, label='Validation')
plt.axhline(noise_level, color='black', ls=':', label='Noise level')
plt.yscale('log')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE (normalized units)')
plt.legend()
plt.show()

In [ ]:
# Predictions on a regular grid of redshifts
z_grid = np.linspace(0, 3, 200).reshape(-1, 1)
preds = predict(network, (z_grid - X_mean) / X_std)

# Denormalize predictions
preds = preds * y_std + y_mean

plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, s=10, label='Training data', color='gray', alpha=0.5)
plt.scatter(X_val, y_val, s=10, label='Validation data', color='tab:blue', alpha=0.5)
plt.plot(z_grid, preds, label='Prediction', color='red', lw=2)
plt.plot(z_grid, true_relation(z_grid), label='True relation', color='black', ls='--')
plt.xlabel('Redshift (z)')
plt.ylabel('Distance (Mpc)')
plt.title('MLP Prediction vs True Relation')
plt.legend()
plt.show()

## 4. Comparing Optimizers

All optimizers get exactly the same gradients, they only differ in the update rule. To compare them fairly, we start from the same initial weights and use the same learning rate.

With momentum, gradients which point in the same direction for many steps add up: for $\gamma = 0.9$ the effective step size becomes up to $1/(1-\gamma) = 10$ times larger, so the loss drops about ten times earlier than for plain SGD (and overshoots a bit on the way, which causes the wiggles). Adam normalizes the step of each parameter by the typical size of its gradient, so every parameter moves by roughly the learning rate per step, no matter how large or small its gradient is.

There is no universally best optimizer and the picture changes with the learning rate (try `learning_rate = 0.01`). Adam is the most popular default because it usually works well without much tuning.

In [ ]:
learning_rate = 0.001
optimizers = {
    'SGD': SGD(learning_rate=learning_rate),
    'SGD with momentum': SGD(learning_rate=learning_rate, momentum=0.9),
    'Adam': Adam(learning_rate=learning_rate),
}

plt.figure(figsize=(8, 5))
for name, optimizer in optimizers.items():
    network_opt = make_network()          # same initial weights for every optimizer
    train_loss, val_loss = train(network_opt, optimizer, X_train_norm, y_train_norm, X_val_norm, y_val_norm, epochs=2000)
    plt.plot(np.arange(1, len(train_loss) + 1), train_loss, label=name)
plt.axhline(noise_level, color='black', ls=':', label='Noise level')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Training MSE (normalized units)')
plt.title(f'Optimizer comparison (learning rate {learning_rate})')
plt.legend()
plt.show()

## 5. Overfitting and Regularization

With 200 training points and a small network there is not much room for overfitting. To see it, we make the problem harder: we keep only 15 training points and use a much larger network (64 hidden units per layer, more than 4000 weights).

We train this network three times: without regularization, with L2 and with L1. Things to look at:
- **Fits:** without regularization the network starts to follow the noise of the individual points.
- **Losses:** the training loss keeps decreasing, but the validation loss goes up again. This is the signature of overfitting, and it is the reason why we always monitor a validation loss.
- **Weights:** L2 shrinks all weights towards zero. L1 sets most of them to (almost exactly) zero and keeps only a few large ones, i.e. it produces a *sparse* network.

In [ ]:
n_small = 15
X_small, y_small = X_train_norm[:n_small], y_train_norm[:n_small]

settings = {
    'No regularization': dict(),
    r'L2 ($\lambda_2 = 0.01$)': dict(l2_lambda=0.01),
    r'L1 ($\lambda_1 = 0.003$)': dict(l1_lambda=0.003),
}

fig, axs = plt.subplots(1, 3, figsize=(18, 5))
axs[0].scatter(X_train[:n_small], y_train[:n_small], s=25, color='black', zorder=3, label='Training data')
axs[0].plot(z_grid, true_relation(z_grid), 'k--', label='True relation')

for name, regularization in settings.items():
    network_reg = make_network(hidden_dim=64)
    optimizer = SGD(learning_rate=0.01, momentum=0.9, **regularization)
    train_loss, val_loss = train(network_reg, optimizer, X_small, y_small, X_val_norm, y_val_norm, epochs=10000)
    print(f'{name:26s} training loss: {train_loss[-1]:.4f}, validation loss: {val_loss[-1]:.4f}')

    preds = predict(network_reg, (z_grid - X_mean) / X_std) * y_std + y_mean
    line, = axs[0].plot(z_grid, preds, label=name)
    axs[1].plot(train_loss, color=line.get_color(), ls=':')
    axs[1].plot(val_loss, color=line.get_color(), label=name)
    # Weights of the large hidden layer (64 x 64)
    axs[2].hist(network_reg[2].weights.ravel(), bins=np.linspace(-0.6, 0.6, 121),
                histtype='step', color=line.get_color(), label=name)

axs[0].set(xlabel='Redshift (z)', ylabel='Distance (Mpc)', title='Fits')
axs[1].set(xlabel='Epoch', ylabel='MSE (normalized units)', xscale='log', yscale='log', xlim=(10, None), ylim=(3e-3, 0.1),
           title='Validation loss (solid) and training loss (dotted)')
axs[2].set(xlabel='Weight', ylabel='Number of weights', yscale='log', title='Weights of the hidden layer')
for ax in axs:
    ax.legend()
plt.show()

## Summary

- A neural network is a chain of simple layers. Each layer only needs a `forward` method and a `backward` method which applies the chain rule locally. Backpropagation is nothing but calling `backward` in reverse order.
- Analytic gradients are easy to get wrong. A numerical gradient check is a cheap way to make sure they are right.
- All optimizers use the same gradients and only differ in how they turn them into an update. Momentum averages the gradients over time, Adam additionally adapts the step size of each parameter.
- The training loss alone does not tell us how good a model is. The loss on independent validation data does: when it starts to increase while the training loss keeps decreasing, the model is overfitting. Regularization (or more data, a smaller network, stopping early) helps.

**Next:** in Notebook 2 we let PyTorch do all of this for us. Every piece we wrote by hand here has a direct counterpart there.